In [14]:
import re
import nltk
import pandas as pd
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

lemmatizer = WordNetLemmatizer()

In [2]:
def clean_text(text: str) -> str:
    """Cleans review text while preserving negation words like 'not' and 'don't'."""
    if not isinstance(text, str):
        return ""
    
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|<.*?>", "", text)
    # Keep letters and apostrophes so "don't" stays intact
    text = re.sub(r"[^a-z\s']", " ", text)
    
    # Lemmatise words (reduces vocabulary noise without losing meaning)
    words = [lemmatizer.lemmatize(w) for w in text.split() if len(w) > 1]
    return " ".join(words)

In [6]:
df = pd.read_csv("../data/flipkart_reviews.csv", encoding="latin1", encoding_errors="replace")
df.columns = [c.strip().lower() for c in df.columns]
df.columns

Index(['productname', 'price', 'rate', 'review', 'summary'], dtype='str')

In [8]:
# Combine review summary/title and detailed review text
title_col = "title" if "title" in df.columns else df.columns[0]
review_col = "review" if "review" in df.columns else df.columns[1]
rating_col = "rating" if "rating" in df.columns else "rate"

In [9]:
df["full_text"] = (df[title_col].fillna("") + " " + df[review_col].fillna("")).str.strip()
df[rating_col] = pd.to_numeric(df[rating_col], errors="coerce")
df = df.dropna(subset=[rating_col])

In [10]:
df.head()

,productname,price,rate,review,summary,full_text
0,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5.0,Super!,Great cooler.. excellent air flow and for this...,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...
1,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5.0,Awesome,Best budget 2 fit cooler. Nice cooling,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...
2,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3.0,Fair,The quality is good but the power of air is de...,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...
3,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",1.0,Useless product,Very bad product it's a only a fan,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...
4,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3.0,Fair,Ok ok product,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...


In [11]:
# Remove empty rows and exact duplicate reviews to prevent data leakage
df = df[df["full_text"].str.len() > 3].drop_duplicates(subset=["full_text"])

In [12]:
def label_mapper(r):
        if r <= 2:
            return "Negative"
        elif r == 3:
            return "Neutral"
        return "Positive"

In [15]:
df["sentiment"] = df[rating_col].apply(label_mapper)
df["cleaned_text"] = df["full_text"].apply(clean_text)

In [16]:
df["sentiment"].value_counts()

sentiment
Positive    16203
Negative     7392
Neutral      2948
Name: count, dtype: int64

In [17]:
df.head()

,productname,price,rate,review,summary,full_text,sentiment,cleaned_text
0,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5.0,Super!,Great cooler.. excellent air flow and for this...,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,Positive,candes room personal air cooler white black el...
1,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5.0,Awesome,Best budget 2 fit cooler. Nice cooling,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,Positive,candes room personal air cooler white black el...
2,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",3.0,Fair,The quality is good but the power of air is de...,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,Neutral,candes room personal air cooler white black el...
3,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",1.0,Useless product,Very bad product it's a only a fan,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,Negative,candes room personal air cooler white black el...
6,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,"??3,999",5.0,Highly recommended,Very good product,Candes 12 L Room/Personal Air Cooler?ÿ?ÿ(White...,Positive,candes room personal air cooler white black el...
